# LC 547 — Number of Provinces
**Day 73 · Union-Find · Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Each city starts as its own province.
Union-Find merges connected cities. The answer is the number of
nodes that are still their own root after processing all connections.
</div>

## Official Problem Statement

There are `n` cities. Some of them are connected directly; others are
connected indirectly. If city `a` is connected to `b`, and `b` is
connected to `c`, then cities `a`, `b`, and `c` are in one province.

You are given an `n x n` matrix `isConnected` where
`isConnected[i][j] == 1` means city `i` and city `j` are directly
connected.

Return the **total number of provinces**.

**Constraints:**
- `1 <= n <= 200`
- `isConnected[i][i] == 1`
- `isConnected[i][j] == isConnected[j][i]`

## What This Is Actually Asking

Think of cities as dots and connections as strings between them.
A province is a cluster of dots you can reach by following any
string. We want to count how many separate clusters exist.

The adjacency matrix just spells out every direct connection:
row `i`, column `j` being `1` means there's a string between
city `i` and city `j`.

We only need to look at the **upper triangle** of the matrix
(or lower) because the matrix is symmetric — we'd double-count
otherwise.

## Walk Through an Example by Hand

```
isConnected = [[1,1,0],
               [1,1,0],
               [0,0,1]]
n = 3 cities: 0, 1, 2
```

**Step 1 — initialise:**
```
parent = [0, 1, 2]   rank = [0, 0, 0]
provinces = 3
```

**Step 2 — scan upper triangle (i < j):**
```
i=0, j=1: isConnected[0][1]=1 → union(0,1)
  find(0)=0, find(1)=1, different roots
  parent[1]=0, rank[0]+=1 → parent=[0,0,2], rank=[1,0,0]
  provinces = 2

i=0, j=2: isConnected[0][2]=0 → skip
i=1, j=2: isConnected[1][2]=0 → skip
```

**Result:** provinces = **2** ✓

## The Picture

Union-Find parent array evolving for n=4, edges: (0,1),(1,2),(3,?)

```
Initial state:
  index:  0   1   2   3
  parent: 0   1   2   3    ← every node is its own root
  rank:   0   0   0   0
  provinces = 4

After union(0,1):  (same rank → attach 1 under 0, bump rank[0])
  index:  0   1   2   3
  parent: 0   0   2   3
  rank:   1   0   0   0
  provinces = 3

After union(1,2):  find(1)=0, find(2)=2
  rank[0]=1 > rank[2]=0 → attach 2 under 0
  index:  0   1   2   3
  parent: 0   0   0   3
  rank:   1   0   0   0
  provinces = 2

Path compression demo — find(1):
  parent[1] = 0? yes → return 0  (no compression needed here)
  find(2) before compression: parent[2]=0 → return 0

Final: 2 distinct roots (0 and 3) → 2 provinces
  Roots: node where parent[node] == node
  0 → parent[0]=0 ✓ root
  1 → parent[1]=0   not root
  2 → parent[2]=0   not root
  3 → parent[3]=3 ✓ root
```

## When To Use This Pattern

Use Union-Find when you need to:
- **Count connected components** in a graph
- **Detect cycles** in an undirected graph
- **Dynamically merge** groups and query membership
- Answer "are these two nodes in the same group?"

**Signals in the problem:**
- Words like *province*, *island*, *cluster*, *group*, *component*
- Adjacency matrix given (not adjacency list)
- Transitive connectivity: if A-B and B-C, then A and C are grouped

**Union-Find vs BFS/DFS:** Both work here. Union-Find is preferred
when edges arrive dynamically or when you need near-O(1) merges.

## The Approach

1. **Initialise** `parent[i] = i` and `rank[i] = 0` for all i.
   Set `provinces = n`.

2. **find(x)** with path compression:
   - If `parent[x] != x`: `parent[x] = find(parent[x])`
   - Return `parent[x]`

3. **union(a, b)** with union by rank:
   - `ra, rb = find(a), find(b)`
   - If `ra == rb`: already same group, return
   - If `rank[ra] < rank[rb]`: swap ra, rb
   - `parent[rb] = ra`
   - If `rank[ra] == rank[rb]`: `rank[ra] += 1`
   - Decrement `provinces`

4. **Scan upper triangle** of `isConnected`:
   - For `i` in range(n), for `j` in range(i+1, n):
     - If `isConnected[i][j] == 1`: `union(i, j)`

5. Return `provinces`.

In [ ]:
from typing import List

In [ ]:
def test_harness(func):
    """
    Run test cases for findCircleNum.
    Prints PASSED / FAILED and a summary line.
    """
    tests = [
        # (isConnected, expected)
        ([[1,1,0],[1,1,0],[0,0,1]], 2),
        ([[1,0,0],[0,1,0],[0,0,1]], 3),
        ([[1,1,1],[1,1,1],[1,1,1]], 1),
        ([[1]],                     1),
        (
            [[1,0,0,1],
             [0,1,1,0],
             [0,1,1,0],
             [1,0,0,1]],
            2
        ),
    ]
    passed = 0
    for i, (grid, expected) in enumerate(tests):
        result = func(grid)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(
            f"Test {i+1}: {status} "
            f"(got {result}, expected {expected})"
        )
    total = len(tests)
    print(f"\nResult: {passed}/{total} tests passed.")

In [ ]:
def findCircleNum(isConnected: List[List[int]]) -> int:
    """
    LC 547 — Number of Provinces.

    Use Union-Find to count connected components.

    Args:
        isConnected: n x n symmetric adjacency matrix.
            isConnected[i][j] == 1 means city i and j
            are directly connected.

    Returns:
        Number of provinces (connected components).

    Examples:
        >>> findCircleNum([[1,1,0],[1,1,0],[0,0,1]])
        2
        >>> findCircleNum([[1,0,0],[0,1,0],[0,0,1]])
        3

    Plan:
        1. parent[i]=i, rank[i]=0, provinces=n
        2. find(x) with path compression
        3. union(a,b) with union by rank; decrement provinces
        4. Scan upper triangle; union where connected
        5. Return provinces
    """
    n = len(isConnected)
    parent = list(range(n))
    rank = [0] * n
    provinces = n

    # Debug: initial state
    print(f"[DEBUG] n={n}, initial provinces={provinces}")

    def find(x: int) -> int:
        """Path-compressed find."""
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(a: int, b: int) -> None:
        nonlocal provinces
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if rank[ra] < rank[rb]:
            ra, rb = rb, ra
        parent[rb] = ra
        if rank[ra] == rank[rb]:
            rank[ra] += 1
        provinces -= 1
        print(
            f"  [DEBUG] union({a},{b}) "
            f"→ parent={parent}, provinces={provinces}"
        )

    pass  # TODO: scan upper triangle and call union

    print(f"[DEBUG] final parent={parent}")
    return provinces

In [ ]:
# Uncomment and run when solution is ready
# test_harness(findCircleNum)

## Complexity

| | Value |
|---|---|
| **Time** | O(n²·α(n)) — scan all n² matrix entries; each union/find is near O(1) with path compression + union by rank (α = inverse Ackermann, effectively constant) |
| **Space** | O(n) — `parent` and `rank` arrays of size n |

**Note:** The bottleneck is reading the n×n matrix, not the
Union-Find operations themselves.

## Real World Connection

**Network segmentation in cloud infrastructure:**
Imagine each city is a VPC (Virtual Private Cloud) and each
direct connection is a VPC peering link. The number of provinces
tells you how many isolated network segments exist.

AWS uses exactly this logic internally when validating that
peering routes don't create unintended full-mesh connectivity.
A security team might run a Union-Find scan nightly to detect
if a newly added peering accidentally merged two segments that
should remain isolated (e.g., prod and dev environments).

> **Simplicity and clarity is Gold.** — Sean's Study Mantra